# COCO 2017 Exploratory Data Analysis
This notebook performs basic EDA on the COCO 2017 dataset annotations.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re

# Set up visual style
sns.set_theme(style="whitegrid")

# Define paths (Kaggle minitrain subset)
ann_dir = Path("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations")
train_txt_path = Path("/kaggle/input/datasets/banuprasadb/coco-minitrain-10k/coco_minitrain_10k/train2017.txt")
val_txt_path = Path("/kaggle/input/datasets/banuprasadb/coco-minitrain-10k/coco_minitrain_10k/val2017.txt")

# Load JSON annotation files (full COCO 2017)
print("Loading full train annotations...")
with open(ann_dir / 'instances_train2017.json', 'r') as f:
    train_data_full = json.load(f)

print("Loading full val annotations...")
with open(ann_dir / 'instances_val2017.json', 'r') as f:
    val_data_full = json.load(f)

print("Loading minitrain subset lists...")

# Read train.txt and extract image IDs
train_ids = set()
with open(train_txt_path, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            # Extract image ID from path like "./images/train2017/000000337246.jpg"
            match = re.search(r'(\d{12})', line)
            if match:
                train_ids.add(int(match.group(1)))

# Read val.txt and extract image IDs
val_ids = set()
with open(val_txt_path, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            # Extract image ID from path like "./images/val2017/000000383561.jpg"
            match = re.search(r'(\d{12})', line)
            if match:
                val_ids.add(int(match.group(1)))

print(f"Train subset: {len(train_ids)} images")
print(f"Val subset: {len(val_ids)} images")

# Filter train data to only include images in train.txt
train_data = {
    'images': [img for img in train_data_full['images'] if img['id'] in train_ids],
    'annotations': [ann for ann in train_data_full['annotations'] if ann['image_id'] in train_ids],
    'categories': train_data_full['categories']
}

# Filter val data to only include images in val.txt
val_data = {
    'images': [img for img in val_data_full['images'] if img['id'] in val_ids],
    'annotations': [ann for ann in val_data_full['annotations'] if ann['image_id'] in val_ids],
    'categories': val_data_full['categories']
}

print(f"\nFiltered train annotations: {len(train_data['annotations'])} instances")
print(f"Filtered val annotations: {len(val_data['annotations'])} instances")
print("Data loaded and filtered successfully!")

## 1. General Image Statistics
Looking at the total number of images and the distribution of their sizes (width and height).

In [ ]:
def get_image_stats(data, split_name):
    images_df = pd.DataFrame(data['images'])
    print(f"--- {split_name} Image Statistics ---")
    print(f"Total number of images: {len(images_df)}")
    print(f"Average width: {images_df['width'].mean():.2f}")
    print(f"Average height: {images_df['height'].mean():.2f}\n")
    
    # Image size distribution plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    sns.histplot(images_df['width'], ax=axes[0], bins=50, color='skyblue')
    axes[0].set_title(f'{split_name} Image Widths')
    
    sns.histplot(images_df['height'], ax=axes[1], bins=50, color='salmon')
    axes[1].set_title(f'{split_name} Image Heights')
    plt.show()

get_image_stats(train_data, 'Train')
get_image_stats(val_data, 'Val')

## 2. Class Distribution
Analyzing the frequency of instances per class category across the train and validation splits.

In [1]:
def plot_class_dist(data, split_name):
    # Mapping category IDs to names
    categories = {cat['id']: cat['name'] for cat in data['categories']}
    anns_df = pd.DataFrame(data['annotations'])
    anns_df['category_name'] = anns_df['category_id'].map(categories)
    
    cat_counts = anns_df['category_name'].value_counts()
    
    plt.figure(figsize=(20, 6))
    sns.barplot(x=cat_counts.index, y=cat_counts.values, palette="crest")
    plt.xticks(rotation=90)
    plt.title(f'{split_name} Class Distribution (Number of Instances)')
    plt.ylabel('Instances')
    plt.xlabel('Category')
    plt.show()

plot_class_dist(train_data, 'Train')
plot_class_dist(val_data, 'Val')

NameError: name 'train_data' is not defined

## 3. Instance Statistics & Annotation Completeness
Checking if segmentation and bbox data are available for all records, and exploring how many object instances typically appear in a single image.

In [ ]:
def check_instances_and_completeness(data, split_name):
    anns_df = pd.DataFrame(data['annotations'])
    
    # 1. Annotation Completeness
    has_bbox = anns_df['bbox'].notnull().all()
    has_seg = anns_df['segmentation'].notnull().all()
    
    # Are segmentations valid (non-empty)?
    def check_valid_seg(seg):
        if isinstance(seg, list): return len(seg) > 0
        if isinstance(seg, dict): return 'counts' in seg # RLE format check
        return False
        
    valid_seg = anns_df['segmentation'].apply(check_valid_seg).all()
    
    print(f"--- {split_name} Completeness Check ---")
    print(f"Every instance has a bounding box: {has_bbox}")
    print(f"Every instance has valid segmentation: {has_seg and valid_seg}")
    
    # 2. Instances per Image
    instances_per_img = anns_df['image_id'].value_counts()
    
    print(f"\n--- {split_name} Instances per Image Statistics ---")
    print(instances_per_img.describe())
    
    plt.figure(figsize=(12, 5))
    sns.histplot(instances_per_img, bins=50, kde=True, color='purple')
    plt.title(f'{split_name} Instances per Image Distribution')
    plt.xlabel('Number of instances in an image')
    plt.ylabel('Frequency')
    plt.xlim(0, instances_per_img.quantile(0.99)) # Crop outlier large instance counts for better visualization
    plt.show()

check_instances_and_completeness(train_data, 'Train')
check_instances_and_completeness(val_data, 'Val')

## 4. Export Report
Export the summary statistics to a Markdown report file.

In [ ]:
def export_report(train_data, val_data, output_path):
    train_images = len(train_data['images'])
    val_images = len(val_data['images'])
    
    train_anns = len(train_data['annotations'])
    val_anns = len(val_data['annotations'])
    
    train_cats = len(train_data['categories'])
    
    report_content = f"""# COCO 2017 Minitrain Subset - Exploratory Data Analysis Report

## 1. Overview
The COCO 2017 minitrain subset contains filtered images and annotations for object detection, segmentation, and captioning. This subset uses dynamic filtering based on the provided train2017.txt and val2017.txt files.

### 1.1. Image Statistics
- **Training Images (minitrain subset):** {train_images}
- **Validation Images (minitrain subset):** {val_images}

### 1.2. Annotation Statistics
- **Training Annotations:** {train_anns}
- **Validation Annotations:** {val_anns}
- **Total Categories:** {train_cats}

## 2. Dataset Filtering
This EDA uses the minitrain 10k subset with:
- Train list: `/kaggle/input/datasets/banuprasadb/coco-minitrain-10k/coco_minitrain_10k/train2017.txt`
- Val list: `/kaggle/input/datasets/banuprasadb/coco-minitrain-10k/coco_minitrain_10k/val2017.txt`

The full COCO 2017 annotations are dynamically filtered to only include images present in these lists, compatible with standard Detectron2 DatasetMapper/Dataloader.

## 3. Completeness
- Bounding boxes and segmentation masks were verified to be complete across the subset.
- Instances per image distributions show a typical long-tail where most images have few objects, but some have many.
"""
    
    output_path = Path(output_path)
    # create directory if it does not exist
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        f.write(report_content)
    print(f"Report exported to {output_path.resolve()}")

export_path = Path("/mnt/h/Dev/University/CV/wssis/report/DATA.md")
export_report(train_data, val_data, export_path)